# Liveness 01 · Dataset Preparation

| Dataset | Size | Attacks | Role |
|---|---|---|---|
| **CelebA-Spoof** | 625 k images, 10 k subjects | print, replay, 3D mask, paper cut | main training set |
| **OULU-NPU** | 4 950 videos, 6 phones | print, replay | train + protocol tests |
| **SiW / SiW-Mv2** | 4.5 k / 1.7 k videos | print, replay, masks, makeup | train (masks, makeup) |
| **Replay-Attack**, **MSU-MFSD** | small | print, replay | **cross-dataset test only** |
| **In-house** (consented) | target 30 k frames | printed photo, phone + tablet replay in OUR garages | fine-tune + final test |

All research licences. Build one manifest with `source, subject, label (1 live / 0 spoof), attack_type, split`.
**Cross-dataset protocol:** train on CelebA-Spoof + OULU (+SiW), test on MSU + Replay → honest generalisation number.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Liveness
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import pandas as pd
LIVE = DATA / "Liveness"
LIVE.mkdir(parents=True, exist_ok=True)
TEST_ONLY_SOURCES = {"msu_mfsd", "replay_attack"}

def manifest_celeba_spoof(root):
    rows = []
    for img in Path(root).rglob("*.jpg"):
        parts = img.parts
        label = 1 if "live" in parts else 0
        rows.append({"path": str(img), "source": "celeba_spoof", "subject": img.parent.parent.name,
                     "label": label, "attack_type": "none" if label else "print_or_replay"})
    return pd.DataFrame(rows)

frames = []
if (LIVE / "CelebA_Spoof").exists():
    frames.append(manifest_celeba_spoof(LIVE / "CelebA_Spoof"))
# frames.append(manifest_oulu(...)); frames.append(manifest_siw(...)); frames.append(manifest_msu(...)) ...
manifest = pd.concat(frames) if frames else pd.DataFrame(columns=["path", "source", "subject", "label", "attack_type"])

In [ ]:
from Common_Utils.Data_Utils import stable_split
manifest["split"] = [("test" if src in TEST_ONLY_SOURCES else stable_split(f"{src}:{subj}"))
                     for src, subj in zip(manifest["source"], manifest["subject"])]
manifest.to_csv(LIVE / "Manifest.csv", index=False)
manifest.groupby(["split", "label"]).size() if len(manifest) else "no data yet"